In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False
# filtered_series_dict: 包含每只股票的 original, filtered, group, params

def build_dsp_factor(original, filtered, lookback=60):
    """
    构建DSP增强因子
    
    因子逻辑：
    - 分子：滤波后价格的动量（趋势强度）
    - 分母：噪声强度（残差的滚动波动率）
    - 因子值高 = 趋势强 + 噪声低 = 高质量买入信号
    
    参数:
    - original: 原始价格序列
    - filtered: 滤波后价格序列
    - lookback: 动量计算周期（60天）
    
    返回:
    - dsp_factor: DSP增强因子
    - filtered_momentum: 滤波后动量
    - original_momentum: 原始动量（对比用）
    - noise_intensity: 噪声强度
    """
    # 对齐数据
    common_idx = original.index.intersection(filtered.index)
    orig_aligned = original.loc[common_idx]
    filt_aligned = filtered.loc[common_idx]
    
    # 1. 滤波后动量（趋势强度）
    filtered_momentum = filt_aligned.pct_change(periods=lookback)
    
    # 2. 噪声强度（残差的滚动波动率）
    residual = orig_aligned.values - filt_aligned.values
    residual_series = pd.Series(residual, index=common_idx)
    noise_intensity = residual_series.rolling(window=lookback).std()
    
    # 3. DSP增强因子 = 动量 / 噪声强度
    dsp_factor = filtered_momentum / (noise_intensity + 1e-8)
    
    # 4. 原始动量（作为对比基准）
    original_momentum = orig_aligned.pct_change(periods=lookback)
    
    return {
        'dsp_factor': dsp_factor,
        'filtered_momentum': filtered_momentum,
        'original_momentum': original_momentum,
        'noise_intensity': noise_intensity
    }

# 1. 读取总表
df_all = pd.read_csv(r"..\数据\全股票滤波结果总表.csv")

df_all['trade_date'] = pd.to_datetime(df_all['trade_date'])


# 3. 按股票代码分组遍历
factors_dict = {}
factor_rows = []  # 用于保存总表
for code, df_stock in df_all.groupby('code'):
    # 取出当前股票：原始收盘价 + 滤波后价格
    original = df_stock['close_price']    # 正确字段名
    filtered = df_stock['filtered_price'] # 正确字段名
    
    # 构建DSP因子
    factors = build_dsp_factor(original, filtered, lookback=60)
    factors_dict[code] = factors
    
     # 拼合成一行行数据，方便存 CSV
    temp = pd.DataFrame({
        'trade_date': df_stock['trade_date'], 
        'code': code,
        'close_price': original,
        'filtered_price': filtered,
        'dsp_raw': factors['dsp_factor'],
        'filtered_momentum': factors['filtered_momentum'],
        'original_momentum': factors['original_momentum'],
        'noise_intensity': factors['noise_intensity'],
        'industry': df_stock['industry'],  # 行业
        'market_cap': df_stock['market_cap']# 市值
    })
    factor_rows.append(temp)

df_factor_all = pd.concat(factor_rows)
import statsmodels.api as sm#线性回归工具
def neutralize_factor_daily(df):
    """ 每日截面：行业 + 市值 中性化 """
    def _neutralize(group):
        log_cap = np.log(group['market_cap'])
        ind_dummies = pd.get_dummies(group['industry'], drop_first=True)
        X = pd.concat([ind_dummies, log_cap.rename('log_cap')], axis=1)
        X = sm.add_constant(X)
        y = group['dsp_raw']

        try:
            res = sm.OLS(y, X).fit().resid
        except:
            res = y

        group['dsp_factor'] = res  # 最终中性化因子
        return group

    return df.groupby('trade_date', group_keys=False).apply(_neutralize)

# 执行中性化
df_factor_all = neutralize_factor_daily(df_factor_all)

df_final = df_factor_all[[
    'trade_date', 'code', 'close_price', 'filtered_price',
    'dsp_factor', 'filtered_momentum', 'original_momentum', 'noise_intensity'
]].sort_values(['code', 'trade_date'])


# 3. 保存完整因子总表

save_path = r"..\数据\DSP因子总表.csv"
df_factor_all.to_csv(save_path, encoding='utf-8-sig')
print("\n✅ 因子总表已保存至：")
print(save_path)

C:\Users\wangwei\AppData\Local\Temp\ipykernel_18348\3501460894.py:109: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('trade_date', group_keys=False).apply(_neutralize)



✅ 因子总表已保存至：
..\数据\DSP因子总表.csv
